LLM - huggingface LLM (default : gpt-3.5-turbo)
https://docs.llamaindex.ai/en/stable/module_guides/models/llms/usage_custom/

In [1]:
from llama_index.core import PromptTemplate

# Transform a string into input zephyr-specific input
def completion_to_prompt(completion):
    return f"<|system|>\n</s>\n<|user|>\n{completion}</s>\n<|assistant|>\n"


# Transform a list of chat messages into zephyr-specific input
def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|system|>\n{message.content}</s>\n"
        elif message.role == "user":
            prompt += f"<|user|>\n{message.content}</s>\n"
        elif message.role == "assistant":
            prompt += f"<|assistant|>\n{message.content}</s>\n"

    # ensure we start with a system prompt, insert blank if needed
    if not prompt.startswith("<|system|>\n"):
        prompt = "<|system|>\n</s>\n" + prompt

    # add final assistant prompt
    prompt = prompt + "<|assistant|>\n"
    
    return prompt

In [2]:
import torch
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings

Settings.llm = HuggingFaceLLM(
    model_name="HuggingFaceH4/zephyr-7b-beta",
    tokenizer_name="HuggingFaceH4/zephyr-7b-beta",
    context_window=3900,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.0, "top_k": 50, "top_p": 0.95},
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto",
)

/home/jjh_test/anaconda3/envs/torch2/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

We've detected an older driver with an RTX 4000 series GPU. These drivers have issues with P2P. This can affect the multi-gpu inference when using accelerate device_map.Please make sure to update your driver to the latest version which resolves this.


EMBEDDING MODEL - BAAI (default : text-embedding-ada-002)
https://docs.llamaindex.ai/en/stable/module_guides/models/embeddings/

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5" #여기서 embedding model 수정
)

VECTOR STORE - 수정 X (customize하려면 pinecone 써야함)

In [4]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)

QUERY ENGINE

In [5]:
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

# build index
index = VectorStoreIndex.from_documents(documents)

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=2,
)

# configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
)

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

In [6]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [7]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [8]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [9]:
a_answers=[]

for answer in answers:
    if "True" in answer:
        a_answer = "True"
    else :
        a_answer= "False"
        
    a_answers.append(a_answer)

In [10]:
responses = []

In [11]:
for question in questions:
    query = f"Start the answer with True or False. You do not have to repeat the question. {question}."
    response = query_engine.query(query)
    print(response)
    response_str = str(response)

    if "True" in response_str:
        response_str = "True"
    else:
        response_str = "False"
    responses.append(response_str)

/home/jjh_test/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jjh_test/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:545: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


False. This is not a multiple choice question. It is an introduction to a hypothetical scenario exploring the potential implications of replacing Earth's atmosphere with a breathable liquid following the discovery of a fourth spatial dimension. The query at the end is asking for multiple choice questions related to this scenario, but it is not provided in the given sources.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


True, the discovery of a fourth spatial dimension allows for the theoretical replacement of Earth's atmosphere with a breathable liquid.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


True, the field of physics that suggests the existence of up to 11 dimensions, including the fourth dimension, is string theory.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


False. The concept of replacing Earth's atmosphere with a breathable liquid, driven by the discovery of a fourth spatial dimension, goes beyond the realm of classical mechanics. While classical mechanics is a fundamental branch of physics that describes the motion of objects under the influence of forces, this transformation would require a deeper understanding of theoretical physics, including string theory and higher-dimensional space concepts. Therefore, the answer is False to the query.


KeyboardInterrupt: 

In [ ]:
correct_count = 0
for a_answer, response_str in zip(a_answers, responses):

    if a_answer == response_str:
        correct_count += 1

print(f"correct_count: {correct_count}")

In [ ]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")